In [5]:
import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import snntorch as snn
from snntorch import spikegen
from snntorch import surrogate
from snntorch import utils
from snntorch import functional as SF
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from PIL import Image

In [2]:
DATA_DIR = "../../data"
DATASET_DIR = f"{DATA_DIR}/processed"

In [3]:
import sys
import os

sys.path.append(os.path.abspath("../.."))

In [6]:
from src.models import BasicSNN, BinaryCNN3, BasicSNNRate, BinaryAbstractionCNN, SNNClassifier, BinarizedSpikingNetwork, CustomLoss

In [7]:
base_dir = "../../models/checkpoints"

available_dirs = {
    0 : "cnn",
    1 : "multi_cnn",
    2 : "basic_snn",
    3 : "basic_multisnn",
    4 : "paper_snn",
    5 : "paper_multisnn",
    6 : "binarized_snn",
}

In [8]:
train_df = pd.read_csv(f"{DATASET_DIR}/train1.csv")
train_df_multi = pd.read_csv(f"{DATASET_DIR}/train1_multi.csv")
val_df = pd.read_csv(f"{DATASET_DIR}/val1.csv")
val_df_multi = pd.read_csv(f"{DATASET_DIR}/val1_multi.csv")

In [9]:
# shuffle val and test
val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_df_multi = val_df_multi.sample(frac=1, random_state=42).reset_index(drop=True)

In [10]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            return img, label

class CustomDatasetAbs(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, epsilon=0):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.epsilon = epsilon
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
        
        # Load and binarize image
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)  # Shape: [1, H, W], values 0 or 1
        
        # Generate perturbation mask
        H, W = img.shape[1], img.shape[2]
        perturb_mask = torch.zeros((H, W), dtype=torch.bool)
        if self.epsilon > 0:
            # Randomly select `epsilon` pixels to perturb
            flat_indices = torch.randperm(H * W)[:self.epsilon]
            perturb_mask.view(-1)[flat_indices] = True
        
        # Create interval bounds (lower and upper channels)
        lower = img.squeeze(0).clone()  # Shape: [H, W]
        upper = img.squeeze(0).clone()
        lower[perturb_mask] = 0  # Perturbed pixels: lower bound = 0
        upper[perturb_mask] = 1  # Perturbed pixels: upper bound = 1
        
        # Stack into 2 channels (lower + upper bounds)
        interval_img = torch.stack([lower, upper], dim=0)  # Shape: [2, H, W]
        
        return interval_img, label

In [11]:
LE = LabelEncoder()
LE_multi = LabelEncoder()

LE.fit(train_df["Label"])
LE_multi.fit(train_df_multi["Label"])

# swap classes in the label encoder
swapped_classes = LE.classes_.copy()
swapped_classes[0], swapped_classes[1] = swapped_classes[1], swapped_classes[0]

LE.classes_ = swapped_classes

train_df_encoded = train_df.copy()
train_df_encoded["Label"] = LE.transform(train_df["Label"])

train_df_multi_encoded = train_df_multi.copy()
train_df_multi_encoded["Label"] = LE_multi.transform(train_df_multi["Label"])

val_df_encoded = val_df.copy()
val_df_encoded["Label"] = LE.transform(val_df["Label"])

val_df_multi_encoded = val_df_multi.copy()
val_df_multi_encoded["Label"] = LE_multi.transform(val_df_multi["Label"])


In [12]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

train_dataset = CustomDataset(train_df_encoded, f"{DATASET_DIR}/images", transform=transform)
val_dataset = CustomDataset(val_df_encoded, f"{DATASET_DIR}/images", transform=transform)

train_multi_dataset = CustomDataset(train_df_multi_encoded, f"{DATASET_DIR}/images", transform=transform)
val_multi_dataset = CustomDataset(val_df_multi_encoded, f"{DATASET_DIR}/images", transform=transform)


In [13]:
from torch.utils.data import WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight

labels = train_df_encoded["Label"].values
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

In [15]:
BATCH_SIZE=40

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

train_multi_loader = DataLoader(train_multi_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_multi_loader = DataLoader(val_multi_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [16]:
def calculate_metrics_ann(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

def calculate_class_metrics(y_true, y_pred):
    return {
        "overall_accuracy": accuracy_score(y_true, y_pred),
        "benign_f1": f1_score(y_true, y_pred, pos_label=1),
        "benign_precision": precision_score(y_true, y_pred, pos_label=1),
        "benign_recall": recall_score(y_true, y_pred, pos_label=1),
        "malicious_f1": f1_score(y_true, y_pred, pos_label=0),
        "malicious_precision": precision_score(y_true, y_pred, pos_label=0),
        "malicious_recall": recall_score(y_true, y_pred, pos_label=0),
    }

from sklearn.metrics import precision_recall_curve

def find_optimal_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    optimal_idx = np.argmax(f1_scores)
    return thresholds[optimal_idx]


def calculate_class_metrics_snn(y_true, y_pred):

    benign_precision = precision_score(y_true, y_pred, pos_label=1)
    benign_recall = recall_score(y_true, y_pred, pos_label=1)
    benign_f1 = f1_score(y_true, y_pred, pos_label=1)

    malicious_precision = precision_score(y_true, y_pred, pos_label=0)
    malicious_recall = recall_score(y_true, y_pred, pos_label=0)
    malicious_f1 = f1_score(y_true, y_pred, pos_label=0)



    metrics = {
        'benign': {
            'precision': benign_precision,
            'recall': benign_recall,
            'f1': benign_f1
        },
        'malicious': {
            'precision': malicious_precision,
            'recall': malicious_recall,
            'f1': malicious_f1
        }
    }
    
    return metrics

def calculate_metrics(y_true, y_pred):
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    
    return {"precision": precision, "recall": recall, "f1": f1}

# CNN

## Binary

In [35]:
def train_model_reduceonplateau(model_class, loss_class, train_dataloader, val_dataloader, optimizer, scheduler, num_epochs=50, lr=1e-3, early_stop=10, device="cuda", weights=None, pos_weights=None, model_savepath=None, weight_decay=0.0, debug=False):
    
    if loss_class == nn.BCEWithLogitsLoss:
        criterion = loss_class(pos_weight=pos_weights) if pos_weights is not None else loss_class()
        model = model_class(num_classes=1)

    elif loss_class == nn.CrossEntropyLoss:
        criterion = loss_class(weight=weights) if weights is not None else loss_class()
        model = model_class(num_classes=2)
    else:
        criterion = loss_class()
        model = model_class(num_classes=2)

    model = model.to(device)

    # optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.999))
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.1)

    best_val_f1 = 0  # Track best overall weighted F1 score
    best_epoch = 0
    best_model = None
    early_stop_counter = 0

    best_threshold = 0.5  # For BCEWithLogitsLoss

    history = {
        "overall_accuracy": {"train": [], "val": []},
        "overall_precision": {"train": [], "val": []},
        "overall_recall": {"train": [], "val": []},
        "overall_f1": {"train": [], "val": []},
        "train_loss": [], "val_loss": [],
        "benign_metrics": {
            "train": {"precision": [], "recall": [], "f1": []},
            "val": {"precision": [], "recall": [], "f1": []}
        },
        "malicious_metrics": {
            "train": {"precision": [], "recall": [], "f1": []},
            "val": {"precision": [], "recall": [], "f1": []}
        }
    }
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # Training phase
        model.train()
        train_loss = 0.0
        train_preds, train_true, train_probs = [], [], []

        for images, labels in tqdm(train_dataloader, desc="Training Batches"):
            labels = labels.float().to(device) if loss_class == nn.BCEWithLogitsLoss else labels.long().to(device)
            images = images.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            
            if loss_class == nn.BCEWithLogitsLoss:
                outputs = outputs.squeeze(-1)
            
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            with torch.no_grad():
                if loss_class == nn.CrossEntropyLoss:
                    probs = torch.softmax(outputs, dim=1).cpu().numpy()
                    preds = torch.argmax(outputs, dim=1).cpu().numpy()
                else:
                    probs = torch.sigmoid(outputs).cpu().numpy()
                    preds = (probs > best_threshold).astype(int)

            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            train_probs.extend(probs)

        train_loss /= len(train_dataloader)
        train_metrics = calculate_class_metrics(train_true, train_preds)

        # Validation phase
        val_loss = 0.0
        val_preds, val_true, val_probs = [], [], []

        with torch.no_grad():
            model.eval()
            for images, labels in tqdm(val_dataloader, desc="Validation Batches"):
                labels = labels.float().to(device) if loss_class == nn.BCEWithLogitsLoss else labels.long().to(device)
                images = images.to(device)

                outputs = model(images)
                
                if loss_class == nn.BCEWithLogitsLoss:
                    outputs = outputs.squeeze(-1)
                
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                if loss_class == nn.CrossEntropyLoss:
                    probs = torch.softmax(outputs, dim=1).cpu().numpy()
                    preds = torch.argmax(outputs, dim=1).cpu().numpy()
                else:
                    probs = torch.sigmoid(outputs).cpu().numpy()
                    preds = (probs > best_threshold).astype(int)

                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
                val_probs.extend(probs)

        val_loss /= len(val_dataloader)

        # Calculate optimal threshold (only for BCEWithLogitsLoss)
        if loss_class == nn.BCEWithLogitsLoss:
            optimal_threshold = find_optimal_threshold(val_true, val_probs)
            val_preds = (np.array(val_probs) > optimal_threshold).astype(int)
            if f1_score(val_true, val_preds, average="weighted") > best_val_f1:
                best_val_f1 = f1_score(val_true, val_preds, average="weighted")
                best_threshold = optimal_threshold

        val_metrics = calculate_class_metrics(val_true, val_preds)

        # Compute overall weighted metrics
        overall_f1_train = f1_score(train_true, train_preds, average="weighted")
        overall_f1_val = f1_score(val_true, val_preds, average="weighted")
        overall_precision_train = precision_score(train_true, train_preds, average="weighted")
        overall_precision_val = precision_score(val_true, val_preds, average="weighted")
        overall_recall_train = recall_score(train_true, train_preds, average="weighted")
        overall_recall_val = recall_score(val_true, val_preds, average="weighted")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["overall_accuracy"]["train"].append(train_metrics["overall_accuracy"])
        history["overall_accuracy"]["val"].append(val_metrics["overall_accuracy"])
        history["overall_f1"]["train"].append(overall_f1_train)
        history["overall_f1"]["val"].append(overall_f1_val)
        history["overall_precision"]["train"].append(overall_precision_train)
        history["overall_precision"]["val"].append(overall_precision_val)
        history["overall_recall"]["train"].append(overall_recall_train)
        history["overall_recall"]["val"].append(overall_recall_val)

        for metric in ["precision", "recall", "f1"]:
            history["benign_metrics"]["train"][metric].append(train_metrics[f"benign_{metric}"])
            history["benign_metrics"]["val"][metric].append(val_metrics[f"benign_{metric}"])
            history["malicious_metrics"]["train"][metric].append(train_metrics[f"malicious_{metric}"])
            history["malicious_metrics"]["val"][metric].append(val_metrics[f"malicious_{metric}"])

        print(f"\nLoss - Train: {train_loss:.4f}, Val: {val_loss:.4f}")
        print(f"Overall Accuracy - Train: {train_metrics['overall_accuracy']:.4f}, Val: {val_metrics['overall_accuracy']:.4f}")
        print(f"Overall F1 - Train: {overall_f1_train:.4f}, Val: {overall_f1_val:.4f}")
        print(f"Overall Precision - Train: {overall_precision_train:.4f}, Val: {overall_precision_val:.4f}")
        print(f"Overall Recall - Train: {overall_recall_train:.4f}, Val: {overall_recall_val:.4f}")

        # Early stopping with model saving based on weighted F1
        if overall_f1_val > best_val_f1:
            best_val_f1 = overall_f1_val
            best_epoch = epoch
            best_model = model.state_dict()
            torch.save(model.state_dict(), model_savepath) if model_savepath else None
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if early_stop_counter >= early_stop:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        scheduler.step(val_loss)

    print(f"\nBest model was from epoch {best_epoch+1} with validation weighted F1 {best_val_f1:.4f}")
    return best_model, history, best_threshold if loss_class == nn.BCEWithLogitsLoss else best_model, history


In [ ]:
def objective(trial, model, train_loader, val_loader, model_savepath):
    # Define hyperparameters to tune
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    num_epochs = trial.suggest_int("num_epochs", 20, 100)
    early_stop = trial.suggest_int("early_stop", 5, 20)
    beta1 = trial.suggest_float("beta1", 0.8, 0.999)
    beta2 = trial.suggest_float("beta2", 0.9, 0.9999)
    reduce_lr_factor = trial.suggest_float("reduce_lr_factor", 0.1, 0.5)
    reduce_lr_patience = trial.suggest_int("reduce_lr_patience", 2, 5)


    # Define the optimizer and scheduler
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(beta1, beta2))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=reduce_lr_patience, factor=reduce_lr_factor)


    # Train the model with the suggested hyperparameters
    best_model, history = train_model_reduceonplateau(
        model_class=model.__class__,  # Pass the model class
        loss_class=nn.BCEWithLogitsLoss,  # Binary classification
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        num_epochs=num_epochs,
        optimizer=optimizer,
        scheduler=scheduler,
        lr=lr,
        early_stop=early_stop,
        device="cuda",
        weight_decay=weight_decay,
        model_savepath=model_savepath,
        debug=False,
        weights=class_weights
    )

    # Return the validation weighted F1 score to maximize
    return max(history["overall_f1"]["val"])

# Create an Optuna study
study = optuna.create_study(direction="maximize")

# Use a lambda function to pass additional arguments
study.optimize(
    lambda trial: objective(
        trial,
        model=BinaryCNN3(),  
        train_loader=train_loader, 
        val_loader=val_loader,  
        model_savepath=f"{base_dir}/{available_dirs[0]}/optuna_best_model.pt"
    ),
    n_trials=10
)

# Print the best hyperparameters
print("Best hyperparameters: ", study.best_params)

## Multi-class

In [ ]:
def train_model_reduceonplateau_multi(model_class, loss_class, train_dataloader, val_dataloader, optimizer, scheduler, num_epochs=50, lr=1e-3, early_stop=10, device="cuda", weights=None, pos_weights=None, model_savepath=None, weight_decay=0.0, debug=False):
    model = model_class(num_classes=6).to(device)
    
    if loss_class == nn.BCEWithLogitsLoss:
        if pos_weights is not None:
            criterion = loss_class(pos_weight=pos_weights)
        elif weights is not None:
            criterion = loss_class(weight=weights)
        else:
            criterion = loss_class()
    elif loss_class == nn.CrossEntropyLoss:
        if weights is not None:
            criterion = loss_class(weight=weights)
        else:
            criterion = loss_class()
    else:
        criterion = loss_class()

    # optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.999))
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.1)

    best_val_loss = np.inf
    best_epoch = 0
    best_model = None
    early_stop_counter = 0

    history = {
        "train_loss": [], 
        "val_loss": [],
        "train_metrics": {
            "accuracy": [], 
            "precision": [], 
            "recall": [], 
            "f1": []
        },
        "val_metrics": {
            "accuracy": [], 
            "precision": [], 
            "recall": [], 
            "f1": []
        }
    }
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # Training phase
        model.train()
        train_loss = 0.0
        train_preds = []
        train_true = []

        for images, labels in tqdm(train_dataloader, desc="Training Batches"):
            if loss_class == nn.CrossEntropyLoss:
                labels = labels.long().to(device)
            else:
                labels = labels.float().to(device)
            
            images = images.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if loss_class == nn.CrossEntropyLoss:
                preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
            else:
                probs = torch.sigmoid(outputs).detach().cpu().numpy()
                preds = np.argmax(probs, axis=1)

            true = labels.detach().cpu().numpy()
            train_preds.extend(preds)
            train_true.extend(true)

        train_loss /= len(train_dataloader)
        train_metrics = calculate_metrics_ann(train_true, train_preds)

        # Validation phase
        val_loss = 0.0
        val_preds = []
        val_true = []

        with torch.no_grad():
            model.eval()
            for images, labels in tqdm(val_dataloader, desc="Validation Batches"):
                if loss_class == nn.CrossEntropyLoss:
                    labels = labels.long().to(device)
                else:
                    labels = labels.float().to(device)

                images = images.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                if loss_class == nn.CrossEntropyLoss:
                    preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
                else:
                    probs = torch.sigmoid(outputs).detach().cpu().numpy()
                    preds = np.argmax(probs, axis=1)

                true = labels.detach().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(true)

        val_loss /= len(val_dataloader)
        val_metrics = calculate_metrics_ann(val_true, val_preds)

        # Update history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        
        for metric in ["accuracy", "precision", "recall", "f1"]:
            history["train_metrics"][metric].append(train_metrics[metric])
            history["val_metrics"][metric].append(val_metrics[metric])

        if debug:
            print(f"\nLoss - Train: {train_loss:.4f}, Val: {val_loss:.4f}")
            print("\nTraining Metrics:")
            print(f"Accuracy: {train_metrics['accuracy']:.4f}, Precision: {train_metrics['precision']:.4f}, Recall: {train_metrics['recall']:.4f}, F1: {train_metrics['f1']:.4f}")
            print("\nValidation Metrics:")
            print(f"Accuracy: {val_metrics['accuracy']:.4f}, Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}, F1: {val_metrics['f1']:.4f}")
        else:
            print(f"\nLoss - Train: {train_loss:.4f}, Val: {val_loss:.4f}")
            print(f"Accuracy - Train: {train_metrics['accuracy']:.4f}, Val: {val_metrics['accuracy']:.4f}")

        # Early stopping with model saving
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_model = model.state_dict()
            if model_savepath:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': val_loss,
                    'scheduler_state_dict': scheduler.state_dict(),
                    'learning_rate': scheduler.get_last_lr(),
                    'history': history
                }, model_savepath)
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if early_stop_counter >= early_stop:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        scheduler.step(val_loss)

    history["best_val_loss"] = best_val_loss

    print(f"\nBest model was from epoch {best_epoch+1} with validation loss {best_val_loss:.4f}")
    return best_model, history

In [ ]:
def objective(trial, model_class, train_loader, val_loader, model_savepath):
    # Define hyperparameters to tune
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    num_epochs = trial.suggest_int("num_epochs", 20, 100)
    early_stop = trial.suggest_int("early_stop", 5, 20)
    beta1 = trial.suggest_float("beta1", 0.8, 0.999)
    beta2 = trial.suggest_float("beta2", 0.9, 0.9999)
    reduce_lr_factor = trial.suggest_float("reduce_lr_factor", 0.1, 0.5)
    reduce_lr_patience = trial.suggest_int("reduce_lr_patience", 2, 5)

    model = model_class(num_classes=6)

    # Define the optimizer and scheduler
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(beta1, beta2))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=reduce_lr_patience, factor=reduce_lr_factor)


    # Train the model with the suggested hyperparameters
    best_model, history = train_model_reduceonplateau_multi(
        model_class=model_class,  # Pass the model class
        loss_class=nn.CrossEntropyLoss,  # Binary classification
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        num_epochs=num_epochs,
        optimizer=optimizer,
        scheduler=scheduler,
        lr=lr,
        early_stop=early_stop,
        device="cuda",
        weight_decay=weight_decay,
        model_savepath=model_savepath,
        debug=False
    )

    # Return the validation loss to minimize
    return history["best_val_loss"]

# Create an Optuna study
study = optuna.create_study(direction="minimize")

# Use a lambda function to pass additional arguments
study.optimize(
    lambda trial: objective(
        trial,
        model_class=BinaryCNN3,  
        train_loader=train_multi_loader,
        val_loader=val_multi_loader,
        model_savepath=f"{base_dir}/{available_dirs[1]}/optuna_best_model.pt"
    ),
    n_trials=10
)

# Print the best hyperparameters
print("Best hyperparameters: ", study.best_params)

# SNN

## BasicSNN

In [17]:
def train_model(model, optimizer, scheduler, train_dataloader, val_dataloader, loss_fn, lr=1e-3, weight_decay=None, num_epochs=50, model_savepath=None, device="cuda", dt_ms=1.0):
    # Initialize model
    model  = model.to(device)

    # model = BasicSNN(num_steps=4).to(device)
    
    # # Optimizer
    # optimizer = torch.optim.AdamW(model.parameters(), lr=lr, 
    #                              weight_decay=weight_decay if weight_decay else 0,
    #                              betas=(0.9, 0.999))
    
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs // 5, T_mult=1, eta_min=1e-6, last_epoch=-1)

    # Initialize metrics tracking dictionary
    metrics = {
        'train': {
            'loss': [],           # Per batch loss
            'avg_loss': [],       # Per epoch average loss
            'acc': [],            # Per epoch accuracy
            'f1': [],             # Per epoch overall F1
            'class_metrics': [],  # Per epoch class-specific metrics
        },
        'val': {
            'loss': [],
            'avg_loss': [],
            'acc': [],
            'f1': [],
            'class_metrics': [],
        },
        # Spike statistics
        'spike_stats': {
            'train': {
                'avg_spikes_per_neuron': [],
                'spike_rate_hz': [],
                'spike_count': [],
                'active_neurons_percent': [],
                'threshold_proximity_avg': [],
                'threshold_proximity_std': [],
                'membrane_potential_avg': [],
                'membrane_potential_std': [],
            },
            'val': {
                'avg_spikes_per_neuron': [],
                'spike_rate_hz': [],
                'spike_count': [],
                'active_neurons_percent': [],
                'threshold_proximity_avg': [],
                'threshold_proximity_std': [],
                'membrane_potential_avg': [],
                'membrane_potential_std': [],
            },
            'firing_rate_stability': [], # General stability metric
        }
    }

    best_val_f1 = -1.0  # Track best F1 score instead of loss
    best_model_state = None
    neuron_cache = {'layer1': None, 'layer2': None, 'output': None}

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # --- TRAINING PHASE ---
        model.train()
        # Per-epoch tracking
        epoch_data = {
            'train_loss': 0,
            'train_correct': 0.0,
            'train_total': 0,
            'spike_count': 0,
            'active_neurons': 0,
            'total_neurons': 0,
            'total_possible': 0,
            'train_preds': [],
            'train_targets': [],
            'sum_mem': 0.0,
            'sum_sq_mem': 0.0,
            'total_mem_samples': 0,
            'proximity_sum': 0.0,
            'proximity_sq_sum': 0.0,
            'proximity_samples': 0,
        }

        for data, targets in tqdm(train_dataloader, desc="Training"):
            data, targets = data.to(device), targets.to(device)
            utils.reset(model)

            # Forward pass
            spk_rec, mem_rec = model(data)

            # Calculate neuron counts once
            if neuron_cache['layer1'] is None:
                with torch.no_grad():
                    neuron_cache['layer1'] = model.spk1[0, 0].numel()
                    neuron_cache['layer2'] = model.spk2[0, 0].numel()
                    neuron_cache['output'] = spk_rec.size(-1)

            # Loss calculation
            loss = loss_fn(spk_rec, targets)
            epoch_data['train_loss'] += loss.item()
            metrics['train']['loss'].append(loss.item())

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # --- METRICS CALCULATION ---
            with torch.no_grad():
                # Accuracy
                acc = SF.accuracy_rate(spk_rec, targets)
                epoch_data['train_correct'] += acc * data.size(0)
                epoch_data['train_total'] += data.size(0)

                # Predictions for F1
                preds = torch.argmax(spk_rec.sum(dim=0), dim=1)
                epoch_data['train_preds'].append(preds.cpu())
                epoch_data['train_targets'].append(targets.cpu())

                # Spike statistics
                spike_tensor = spk_rec.detach()
                layer1_spikes = model.spk1.detach()
                layer2_spikes = model.spk2.detach()

                batch_spike_count = (spike_tensor.sum() + layer1_spikes.sum() + layer2_spikes.sum()).item()
                epoch_data['spike_count'] += batch_spike_count

                batch_size = data.size(0)
                time_steps = spike_tensor.size(0)
                total_batch_neurons = (neuron_cache['layer1'] + neuron_cache['layer2'] + neuron_cache['output']) * batch_size * time_steps
                epoch_data['total_neurons'] += total_batch_neurons

                active1 = (layer1_spikes.sum(dim=0) > 0).sum(dim=(1,2,3)).sum().item()
                active2 = (layer2_spikes.sum(dim=0) > 0).sum(dim=(1,2,3)).sum().item()
                active3 = (spike_tensor.sum(dim=0) > 0).sum(dim=1).sum().item()
                epoch_data['active_neurons'] += active1 + active2 + active3
                epoch_data['total_possible'] += (neuron_cache['layer1'] + neuron_cache['layer2'] + neuron_cache['output']) * batch_size

                # Membrane statistics
                if mem_rec is not None:
                    output_layer_mem = mem_rec.detach()

                    layer1_mem = model.mem1.detach()
                    layer2_mem = model.mem2.detach()


                    mem_tensor = torch.cat([output_layer_mem.flatten(), layer1_mem.flatten(), layer2_mem.flatten()])
                    
                    epoch_data['sum_mem'] += mem_tensor.sum().item()
                    epoch_data['sum_sq_mem'] += (mem_tensor**2).sum().item()
                    epoch_data['total_mem_samples'] += mem_tensor.numel()

                    proximity = torch.abs(mem_tensor - 0.3)
                    epoch_data['proximity_sum'] += proximity.sum().item()
                    epoch_data['proximity_sq_sum'] += (proximity**2).sum().item()
                    epoch_data['proximity_samples'] += proximity.numel()

        # --- EPOCH TRAINING METRICS ---
        avg_train_loss = epoch_data['train_loss'] / len(train_dataloader)
        metrics['train']['avg_loss'].append(avg_train_loss)
        
        train_acc = epoch_data['train_correct'] / epoch_data['train_total'] if epoch_data['train_total'] > 0 else 0.0
        metrics['train']['acc'].append(train_acc)

        # Calculate F1 score and class-specific metrics
        train_preds = torch.cat(epoch_data['train_preds']).numpy() if len(epoch_data['train_preds']) > 0 else np.array([])
        train_targets = torch.cat(epoch_data['train_targets']).numpy() if len(epoch_data['train_targets']) > 0 else np.array([])
        
        # Overall F1 score (binary average)
        train_f1 = f1_score(train_targets, train_preds, average='weighted') if len(train_preds) > 0 else 0.0
        metrics['train']['f1'].append(train_f1)
        
        # Calculate class-specific metrics
        if len(train_preds) > 0:
            train_class_metrics = calculate_class_metrics_snn(train_targets, train_preds)
            metrics['train']['class_metrics'].append(train_class_metrics)
        else:
            train_class_metrics = {
                'benign': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0},
                'malicious': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
            }
            metrics['train']['class_metrics'].append(train_class_metrics)

        # Spike metrics
        avg_spikes_per_neuron = epoch_data['spike_count'] / epoch_data['total_neurons'] if epoch_data['total_neurons'] > 0 else 0.0
        active_percent = (epoch_data['active_neurons'] / epoch_data['total_possible']) * 100 if epoch_data['total_possible'] > 0 else 0.0
        
        metrics['spike_stats']['train']['avg_spikes_per_neuron'].append(avg_spikes_per_neuron)
        metrics['spike_stats']['train']['spike_rate_hz'].append(avg_spikes_per_neuron * (1000 / dt_ms))
        metrics['spike_stats']['train']['spike_count'].append(epoch_data['spike_count'])
        metrics['spike_stats']['train']['active_neurons_percent'].append(active_percent)

        # Membrane metrics
        if epoch_data['total_mem_samples'] > 0:
            avg_mem = epoch_data['sum_mem'] / epoch_data['total_mem_samples']
            std_mem = np.sqrt((epoch_data['sum_sq_mem'] / epoch_data['total_mem_samples']) - avg_mem**2)
        else:
            avg_mem = std_mem = 0.0
        metrics['spike_stats']['train']['membrane_potential_avg'].append(avg_mem)
        metrics['spike_stats']['train']['membrane_potential_std'].append(std_mem)

        # Threshold proximity
        if epoch_data['proximity_samples'] > 0:
            avg_prox = epoch_data['proximity_sum'] / epoch_data['proximity_samples']
            std_prox = np.sqrt((epoch_data['proximity_sq_sum'] / epoch_data['proximity_samples']) - avg_prox**2)
        else:
            avg_prox = std_prox = 0.0
        metrics['spike_stats']['train']['threshold_proximity_avg'].append(avg_prox)
        metrics['spike_stats']['train']['threshold_proximity_std'].append(std_prox)

        print(f"Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Benign - P: {train_class_metrics['benign']['precision']:.4f}, R: {train_class_metrics['benign']['recall']:.4f}, F1: {train_class_metrics['benign']['f1']:.4f}")
        print(f"Malicious - P: {train_class_metrics['malicious']['precision']:.4f}, R: {train_class_metrics['malicious']['recall']:.4f}, F1: {train_class_metrics['malicious']['f1']:.4f}")
        print(f"Spikes/Neuron: {avg_spikes_per_neuron:.4f} ({avg_spikes_per_neuron*(1000/dt_ms):.1f}Hz)")
        print(f"Active Neurons: {active_percent:.1f}%")
        print("-" * 50)

        # --- VALIDATION PHASE ---
        with torch.no_grad():
            model.eval()
            # Per-epoch validation tracking
            val_data = {
                'val_loss': 0,
                'val_correct': 0.0,
                'val_total': 0,
                'spike_count': 0,
                'total_neurons': 0,
                'active_neurons': 0,
                'total_possible': 0,
                'val_preds': [],
                'val_targets': [],
                'sum_mem': 0.0,
                'sum_sq_mem': 0.0,
                'total_mem_samples': 0,
                'proximity_sum': 0.0,
                'proximity_sq_sum': 0.0,
                'proximity_samples': 0,
            }
            
            for data, targets in tqdm(val_dataloader, desc="Validation"):
                data, targets = data.to(device), targets.to(device)
                utils.reset(model)

                spk_rec, mem_rec = model(data)

                # Loss and accuracy
                loss = loss_fn(spk_rec, targets)
                val_data['val_loss'] += loss.item()
                metrics['val']['loss'].append(loss.item())

                acc = SF.accuracy_rate(spk_rec, targets)
                val_data['val_correct'] += acc * data.size(0)
                val_data['val_total'] += data.size(0)

                # Predictions for F1
                preds = torch.argmax(spk_rec.sum(dim=0), dim=1)
                val_data['val_preds'].append(preds.cpu())
                val_data['val_targets'].append(targets.cpu())

                # Spike statistics
                spike_tensor = spk_rec.detach()
                layer1_spikes = model.spk1.detach()
                layer2_spikes = model.spk2.detach()

                batch_spike = (spike_tensor.sum() + layer1_spikes.sum() + layer2_spikes.sum()).item()
                val_data['spike_count'] += batch_spike

                batch_size = data.size(0)
                time_steps = spike_tensor.size(0)
                val_data['total_neurons'] += (neuron_cache['layer1'] + neuron_cache['layer2'] + neuron_cache['output']) * batch_size * time_steps

                active1 = (layer1_spikes.sum(dim=0) > 0).sum(dim=(1,2,3)).sum().item()
                active2 = (layer2_spikes.sum(dim=0) > 0).sum(dim=(1,2,3)).sum().item()
                active3 = (spike_tensor.sum(dim=0) > 0).sum(dim=1).sum().item()
                val_data['active_neurons'] += active1 + active2 + active3
                val_data['total_possible'] += (neuron_cache['layer1'] + neuron_cache['layer2'] + neuron_cache['output']) * batch_size

                # Membrane stats
                if mem_rec is not None:
                    output_layer_mem = mem_rec.detach()

                    layer1_mem = model.mem1.detach()
                    layer2_mem = model.mem2.detach()


                    mem_tensor = torch.cat([output_layer_mem.flatten(), layer1_mem.flatten(), layer2_mem.flatten()])

                    val_data['sum_mem'] += mem_tensor.sum().item()
                    val_data['sum_sq_mem'] += (mem_tensor**2).sum().item()
                    val_data['total_mem_samples'] += mem_tensor.numel()

                    proximity = torch.abs(mem_tensor - 0.3)
                    val_data['proximity_sum'] += proximity.sum().item()
                    val_data['proximity_sq_sum'] += (proximity**2).sum().item()
                    val_data['proximity_samples'] += proximity.numel()

        # --- VALIDATION METRICS ---
        avg_val_loss = val_data['val_loss'] / len(val_dataloader)
        val_acc = val_data['val_correct'] / val_data['val_total'] if val_data['val_total'] > 0 else 0.0
        metrics['val']['avg_loss'].append(avg_val_loss)
        metrics['val']['acc'].append(val_acc)

        # Calculate validation F1 and class-specific metrics
        val_preds = torch.cat(val_data['val_preds']).numpy() if len(val_data['val_preds']) > 0 else np.array([])
        val_targets = torch.cat(val_data['val_targets']).numpy() if len(val_data['val_targets']) > 0 else np.array([])
        
        # Overall F1 score (binary average)
        val_f1 = f1_score(val_targets, val_preds, average='weighted') if len(val_preds) > 0 else 0.0
        metrics['val']['f1'].append(val_f1)
        
        # Calculate class-specific metrics
        if len(val_preds) > 0:
            val_class_metrics = calculate_class_metrics_snn(val_targets, val_preds)
            metrics['val']['class_metrics'].append(val_class_metrics)
        else:
            val_class_metrics = {
                'benign': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0},
                'malicious': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
            }
            metrics['val']['class_metrics'].append(val_class_metrics)

        # Spike metrics
        val_avg_spikes = val_data['spike_count'] / val_data['total_neurons'] if val_data['total_neurons'] > 0 else 0.0
        val_active_percent = (val_data['active_neurons'] / val_data['total_possible']) * 100 if val_data['total_possible'] > 0 else 0.0
        
        metrics['spike_stats']['val']['avg_spikes_per_neuron'].append(val_avg_spikes)
        metrics['spike_stats']['val']['spike_rate_hz'].append(val_avg_spikes * (1000 / dt_ms))
        metrics['spike_stats']['val']['spike_count'].append(val_data['spike_count'])
        metrics['spike_stats']['val']['active_neurons_percent'].append(val_active_percent)

        # Membrane metrics
        if val_data['total_mem_samples'] > 0:
            avg_val_mem = val_data['sum_mem'] / val_data['total_mem_samples']
            std_val_mem = np.sqrt((val_data['sum_sq_mem'] / val_data['total_mem_samples']) - avg_val_mem**2)
        else:
            avg_val_mem = std_val_mem = 0.0

        # Threshold proximity
        if val_data['proximity_samples'] > 0:
            avg_val_prox = val_data['proximity_sum'] / val_data['proximity_samples']
            std_val_prox = np.sqrt((val_data['proximity_sq_sum'] / val_data['proximity_samples']) - avg_val_prox**2)
        else:
            avg_val_prox = std_val_prox = 0.0
        metrics['spike_stats']['val']['threshold_proximity_avg'].append(avg_val_prox)
        metrics['spike_stats']['val']['threshold_proximity_std'].append(std_val_prox)

        metrics['spike_stats']['val']['membrane_potential_avg'].append(avg_val_mem)
        metrics['spike_stats']['val']['membrane_potential_std'].append(std_val_mem)

        # --- MODEL CHECKPOINTING ---
        # Keep using overall binary F1 for best model selection (as requested)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'val_loss': avg_val_loss,
                'val_acc': val_acc,
                'val_f1': val_f1,
                'val_class_metrics': val_class_metrics
            }
            print(f"New best validation F1: {best_val_f1:.4f}")

        # --- EPOCH REPORTING ---
        print(f"Val Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
        print(f"Benign - P: {val_class_metrics['benign']['precision']:.4f}, R: {val_class_metrics['benign']['recall']:.4f}, F1: {val_class_metrics['benign']['f1']:.4f}")
        print(f"Malicious - P: {val_class_metrics['malicious']['precision']:.4f}, R: {val_class_metrics['malicious']['recall']:.4f}, F1: {val_class_metrics['malicious']['f1']:.4f}")
        print(f"Spikes/Neuron: {val_avg_spikes:.4f} ({val_avg_spikes*(1000/dt_ms):.1f}Hz)")
        print(f"Active Neurons: Train {active_percent:.1f}% | Val {val_active_percent:.1f}%")
        print(f"Membrane Potential: Train {avg_mem:.4f} ± {std_mem:.4f} | Val {avg_val_mem:.4f} ± {std_val_mem:.4f}")
        print(f"Threshold Proximity: Train {avg_prox:.4f} ± {std_prox:.4f} | Val {avg_val_prox:.4f} ± {std_val_prox:.4f}")
        print("-" * 50)

        # Update scheduler
        scheduler.step()

    # --- FINAL SAVING ---
    metrics['best_val_f1'] = best_val_f1

    if model_savepath:
        # Save final model
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': metrics
        }, model_savepath)

        # Save best model
        if best_model_state:
            best_path = model_savepath.replace(".pt", "_best.pt")
            torch.save(best_model_state, best_path)
            print(f"Best model (F1={best_val_f1:.4f}) saved to {best_path}")

    return model, metrics

In [68]:
def train_model(model, optimizer, scheduler, train_dataloader, val_dataloader, loss_fn, lr=1e-3, weight_decay=None, num_epochs=50, model_savepath=None, device="cuda", dt_ms=1.0):
    # Initialize model
    model = model.to(device)
    
    # Optimizer
    # optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay if weight_decay else 0, betas=(0.9, 0.999))
    
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs // 5, T_mult=1, eta_min=1e-6, last_epoch=-1)

    # Initialize metrics tracking dictionary
    metrics = {
        'train': {
            'loss': [],           # Per batch loss
            'avg_loss': [],       # Per epoch average loss
            'acc': [],            # Per epoch accuracy
            'f1': [],             # Per epoch overall F1
            'precision': [],      # Per epoch precision
            'recall': [],         # Per epoch recall
        },
        'val': {
            'loss': [],
            'avg_loss': [],
            'acc': [],
            'f1': [],
            'precision': [],
            'recall': [],
        },
        # Spike statistics
        'spike_stats': {
            'train': {
                'avg_spikes_per_neuron': [],
                'spike_rate_hz': [],
                'spike_count': [],
                'active_neurons_percent': [],
                'threshold_proximity_avg': [],
                'threshold_proximity_std': [],
                'membrane_potential_avg': [],
                'membrane_potential_std': [],
            },
            'val': {
                'avg_spikes_per_neuron': [],
                'spike_rate_hz': [],
                'spike_count': [],
                'active_neurons_percent': [],
                'threshold_proximity_avg': [],
                'threshold_proximity_std': [],
                'membrane_potential_avg': [],
                'membrane_potential_std': [],
            },
            'firing_rate_stability': [], # General stability metric
        }
    }

    best_val_loss = float('inf')
    best_model_state = None
    neuron_cache = {'layer1': None, 'layer2': None, 'output': None}

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # --- TRAINING PHASE ---
        model.train()
        # Per-epoch tracking
        epoch_data = {
            'train_loss': 0,
            'train_correct': 0.0,
            'train_total': 0,
            'spike_count': 0,
            'active_neurons': 0,
            'total_neurons': 0,
            'total_possible': 0,
            'train_preds': [],
            'train_targets': [],
            'sum_mem': 0.0,
            'sum_sq_mem': 0.0,
            'total_mem_samples': 0,
            'proximity_sum': 0.0,
            'proximity_sq_sum': 0.0,
            'proximity_samples': 0,
        }

        for data, targets in tqdm(train_dataloader, desc="Training"):
            data, targets = data.to(device), targets.to(device)
            utils.reset(model)

            # Forward pass
            spk_rec, mem_rec = model(data)

            # Calculate neuron counts once
            if neuron_cache['layer1'] is None:
                with torch.no_grad():
                    neuron_cache['layer1'] = model.spk1[0, 0].numel()
                    neuron_cache['layer2'] = model.spk2[0, 0].numel()
                    neuron_cache['output'] = spk_rec.size(-1)

            # Loss calculation
            loss = loss_fn(spk_rec, targets)
            epoch_data['train_loss'] += loss.item()
            metrics['train']['loss'].append(loss.item())

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # --- METRICS CALCULATION ---
            with torch.no_grad():
                # Accuracy
                acc = SF.accuracy_rate(spk_rec, targets)
                epoch_data['train_correct'] += acc * data.size(0)
                epoch_data['train_total'] += data.size(0)

                # Predictions for F1
                preds = torch.argmax(spk_rec.sum(dim=0), dim=1)
                epoch_data['train_preds'].append(preds.cpu())
                epoch_data['train_targets'].append(targets.cpu())

                # Spike statistics
                spike_tensor = spk_rec.detach()
                layer1_spikes = model.spk1.detach()
                layer2_spikes = model.spk2.detach()

                batch_spike_count = (spike_tensor.sum() + layer1_spikes.sum() + layer2_spikes.sum()).item()
                epoch_data['spike_count'] += batch_spike_count

                batch_size = data.size(0)
                time_steps = spike_tensor.size(0)
                total_batch_neurons = (neuron_cache['layer1'] + neuron_cache['layer2'] + neuron_cache['output']) * batch_size * time_steps
                epoch_data['total_neurons'] += total_batch_neurons

                active1 = (layer1_spikes.sum(dim=0) > 0).sum(dim=(1,2,3)).sum().item()
                active2 = (layer2_spikes.sum(dim=0) > 0).sum(dim=(1,2,3)).sum().item()
                active3 = (spike_tensor.sum(dim=0) > 0).sum(dim=1).sum().item()
                epoch_data['active_neurons'] += active1 + active2 + active3
                epoch_data['total_possible'] += (neuron_cache['layer1'] + neuron_cache['layer2'] + neuron_cache['output']) * batch_size

                # Membrane statistics
                if mem_rec is not None:
                    output_layer_mem = mem_rec.detach()

                    layer1_mem = model.mem1.detach()
                    layer2_mem = model.mem2.detach()


                    mem_tensor = torch.cat([output_layer_mem.flatten(), layer1_mem.flatten(), layer2_mem.flatten()])
                    
                    epoch_data['sum_mem'] += mem_tensor.sum().item()
                    epoch_data['sum_sq_mem'] += (mem_tensor**2).sum().item()
                    epoch_data['total_mem_samples'] += mem_tensor.numel()

                    proximity = torch.abs(mem_tensor - 0.3)
                    epoch_data['proximity_sum'] += proximity.sum().item()
                    epoch_data['proximity_sq_sum'] += (proximity**2).sum().item()
                    epoch_data['proximity_samples'] += proximity.numel()

        # --- EPOCH TRAINING METRICS ---
        avg_train_loss = epoch_data['train_loss'] / len(train_dataloader)
        metrics['train']['avg_loss'].append(avg_train_loss)
        
        train_acc = epoch_data['train_correct'] / epoch_data['train_total'] if epoch_data['train_total'] > 0 else 0.0
        metrics['train']['acc'].append(train_acc)

        # Calculate F1 score and class-specific metrics
        train_preds = torch.cat(epoch_data['train_preds']).numpy() if len(epoch_data['train_preds']) > 0 else np.array([])
        train_targets = torch.cat(epoch_data['train_targets']).numpy() if len(epoch_data['train_targets']) > 0 else np.array([])
        
        # Overall P-R-F1 (macro average) since this is balanced
        train_metrics = calculate_metrics(train_targets, train_preds)

        train_f1 = train_metrics['f1']

        metrics['train']['precision'].append(train_metrics['precision'])
        metrics['train']['recall'].append(train_metrics['recall'])
        metrics['train']['f1'].append(train_f1)

        # Spike metrics
        avg_spikes_per_neuron = epoch_data['spike_count'] / epoch_data['total_neurons'] if epoch_data['total_neurons'] > 0 else 0.0
        active_percent = (epoch_data['active_neurons'] / epoch_data['total_possible']) * 100 if epoch_data['total_possible'] > 0 else 0.0
        
        metrics['spike_stats']['train']['avg_spikes_per_neuron'].append(avg_spikes_per_neuron)
        metrics['spike_stats']['train']['spike_rate_hz'].append(avg_spikes_per_neuron * (1000 / dt_ms))
        metrics['spike_stats']['train']['spike_count'].append(epoch_data['spike_count'])
        metrics['spike_stats']['train']['active_neurons_percent'].append(active_percent)

        # Membrane metrics
        if epoch_data['total_mem_samples'] > 0:
            avg_mem = epoch_data['sum_mem'] / epoch_data['total_mem_samples']
            std_mem = np.sqrt((epoch_data['sum_sq_mem'] / epoch_data['total_mem_samples']) - avg_mem**2)
        else:
            avg_mem = std_mem = 0.0
        metrics['spike_stats']['train']['membrane_potential_avg'].append(avg_mem)
        metrics['spike_stats']['train']['membrane_potential_std'].append(std_mem)

        # Threshold proximity
        if epoch_data['proximity_samples'] > 0:
            avg_prox = epoch_data['proximity_sum'] / epoch_data['proximity_samples']
            std_prox = np.sqrt((epoch_data['proximity_sq_sum'] / epoch_data['proximity_samples']) - avg_prox**2)
        else:
            avg_prox = std_prox = 0.0
        metrics['spike_stats']['train']['threshold_proximity_avg'].append(avg_prox)
        metrics['spike_stats']['train']['threshold_proximity_std'].append(std_prox)

        print(f"Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Spikes/Neuron: {avg_spikes_per_neuron:.4f} ({avg_spikes_per_neuron*(1000/dt_ms):.1f}Hz)")
        print(f"Active Neurons: {active_percent:.1f}%")
        print("-" * 50)

        # --- VALIDATION PHASE ---
        with torch.no_grad():
            model.eval()
            # Per-epoch validation tracking
            val_data = {
                'val_loss': 0,
                'val_correct': 0.0,
                'val_total': 0,
                'spike_count': 0,
                'total_neurons': 0,
                'active_neurons': 0,
                'total_possible': 0,
                'val_preds': [],
                'val_targets': [],
                'sum_mem': 0.0,
                'sum_sq_mem': 0.0,
                'total_mem_samples': 0,
                'proximity_sum': 0.0,
                'proximity_sq_sum': 0.0,
                'proximity_samples': 0,
            }
            
            for data, targets in tqdm(val_dataloader, desc="Validation"):
                data, targets = data.to(device), targets.to(device)
                utils.reset(model)

                spk_rec, mem_rec = model(data)

                # Loss and accuracy
                loss = loss_fn(spk_rec, targets)
                val_data['val_loss'] += loss.item()
                metrics['val']['loss'].append(loss.item())

                acc = SF.accuracy_rate(spk_rec, targets)
                val_data['val_correct'] += acc * data.size(0)
                val_data['val_total'] += data.size(0)

                # Predictions for F1
                preds = torch.argmax(spk_rec.sum(dim=0), dim=1)
                val_data['val_preds'].append(preds.cpu())
                val_data['val_targets'].append(targets.cpu())

                # Spike statistics
                spike_tensor = spk_rec.detach()
                layer1_spikes = model.spk1.detach()
                layer2_spikes = model.spk2.detach()

                batch_spike = (spike_tensor.sum() + layer1_spikes.sum() + layer2_spikes.sum()).item()
                val_data['spike_count'] += batch_spike

                batch_size = data.size(0)
                time_steps = spike_tensor.size(0)
                val_data['total_neurons'] += (neuron_cache['layer1'] + neuron_cache['layer2'] + neuron_cache['output']) * batch_size * time_steps

                active1 = (layer1_spikes.sum(dim=0) > 0).sum(dim=(1,2,3)).sum().item()
                active2 = (layer2_spikes.sum(dim=0) > 0).sum(dim=(1,2,3)).sum().item()
                active3 = (spike_tensor.sum(dim=0) > 0).sum(dim=1).sum().item()
                val_data['active_neurons'] += active1 + active2 + active3
                val_data['total_possible'] += (neuron_cache['layer1'] + neuron_cache['layer2'] + neuron_cache['output']) * batch_size

                # Membrane stats
                if mem_rec is not None:
                    output_layer_mem = mem_rec.detach()

                    layer1_mem = model.mem1.detach()
                    layer2_mem = model.mem2.detach()


                    mem_tensor = torch.cat([output_layer_mem.flatten(), layer1_mem.flatten(), layer2_mem.flatten()])
                    
                    val_data['sum_mem'] += mem_tensor.sum().item()
                    val_data['sum_sq_mem'] += (mem_tensor**2).sum().item()
                    val_data['total_mem_samples'] += mem_tensor.numel()

                    proximity = torch.abs(mem_tensor - 0.3)
                    val_data['proximity_sum'] += proximity.sum().item()
                    val_data['proximity_sq_sum'] += (proximity**2).sum().item()
                    val_data['proximity_samples'] += proximity.numel()

        # --- VALIDATION METRICS ---
        avg_val_loss = val_data['val_loss'] / len(val_dataloader)
        val_acc = val_data['val_correct'] / val_data['val_total'] if val_data['val_total'] > 0 else 0.0
        metrics['val']['avg_loss'].append(avg_val_loss)
        metrics['val']['acc'].append(val_acc)

        # Calculate validation F1 and class-specific metrics
        val_preds = torch.cat(val_data['val_preds']).numpy() if len(val_data['val_preds']) > 0 else np.array([])
        val_targets = torch.cat(val_data['val_targets']).numpy() if len(val_data['val_targets']) > 0 else np.array([])
        
        # Overall P-R-F1 (macro average) since this is balanced
        val_metrics = calculate_metrics(val_targets, val_preds)

        val_f1 = val_metrics['f1']
        metrics['val']['precision'].append(val_metrics['precision'])
        metrics['val']['recall'].append(val_metrics['recall'])
        metrics['val']['f1'].append(val_f1)



        # Spike metrics
        val_avg_spikes = val_data['spike_count'] / val_data['total_neurons'] if val_data['total_neurons'] > 0 else 0.0
        val_active_percent = (val_data['active_neurons'] / val_data['total_possible']) * 100 if val_data['total_possible'] > 0 else 0.0
        
        metrics['spike_stats']['val']['avg_spikes_per_neuron'].append(val_avg_spikes)
        metrics['spike_stats']['val']['spike_rate_hz'].append(val_avg_spikes * (1000 / dt_ms))
        metrics['spike_stats']['val']['spike_count'].append(val_data['spike_count'])
        metrics['spike_stats']['val']['active_neurons_percent'].append(val_active_percent)

        # Membrane metrics
        if val_data['total_mem_samples'] > 0:
            avg_val_mem = val_data['sum_mem'] / val_data['total_mem_samples']
            std_val_mem = np.sqrt((val_data['sum_sq_mem'] / val_data['total_mem_samples']) - avg_val_mem**2)
        else:
            avg_val_mem = std_val_mem = 0.0

        # Threshold proximity
        if val_data['proximity_samples'] > 0:
            avg_val_prox = val_data['proximity_sum'] / val_data['proximity_samples']
            std_val_prox = np.sqrt((val_data['proximity_sq_sum'] / val_data['proximity_samples']) - avg_val_prox**2)
        else:
            avg_val_prox = std_val_prox = 0.0
        metrics['spike_stats']['val']['threshold_proximity_avg'].append(avg_val_prox)
        metrics['spike_stats']['val']['threshold_proximity_std'].append(std_val_prox)

        metrics['spike_stats']['val']['membrane_potential_avg'].append(avg_val_mem)
        metrics['spike_stats']['val']['membrane_potential_std'].append(std_val_mem)

        # --- MODEL CHECKPOINTING ---
        # Use loss as the multi-class is balanced

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'val_loss': avg_val_loss,
                'val_acc': val_acc,
                'val_f1': val_f1,
            }
            print(f"New best validation F1: {best_val_loss:.4f}")

        # --- EPOCH REPORTING ---
        print(f"Val Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
        print(f"Spikes/Neuron: {val_avg_spikes:.4f} ({val_avg_spikes*(1000/dt_ms):.1f}Hz)")
        print(f"Active Neurons: Train {active_percent:.1f}% | Val {val_active_percent:.1f}%")
        print(f"Membrane Potential: Train {avg_mem:.4f} ± {std_mem:.4f} | Val {avg_val_mem:.4f} ± {std_val_mem:.4f}")
        print(f"Threshold Proximity: Train {avg_prox:.4f} ± {std_prox:.4f} | Val {avg_val_prox:.4f} ± {std_val_prox:.4f}")
        print("-" * 50)

        # Update scheduler
        scheduler.step()

    # --- FINAL SAVING ---
    metrics['best_val_loss'] = best_val_loss

    if model_savepath:
        # Save final model
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': metrics
        }, model_savepath)

        # Save best model
        if best_model_state:
            best_path = model_savepath.replace(".pt", "_best.pt")
            torch.save(best_model_state, best_path)
            print(f"Best model (F1={best_val_loss:.4f}) saved at: {best_path}")

    return model, metrics

In [ ]:
def objective(trial, train_dataloader, val_dataloader, loss_fn, device="cuda", dt_ms=1.0, num_classes=2):
    # Define hyperparameters to optimize
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)  # Learning rate
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)  # Weight decay
    num_epochs = trial.suggest_int("num_epochs", 10, 100)  # Number of epochs
    beta1 = trial.suggest_float("beta1", 0.8, 0.999)
    beta2 = trial.suggest_float("beta2", 0.9, 0.9999)
    beta_model = trial.suggest_float("beta_model", 0.1, 0.9)
    num_steps = trial.suggest_int("num_steps", 2, 8)  # Number of time steps

    model = BasicSNN(num_steps=num_steps, num_classes=num_classes, beta=beta_model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(beta1, beta2))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs // 5, T_mult=1, eta_min=1e-6, last_epoch=-1)


    # Train the model using the provided training function
    model, metrics = train_model(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        loss_fn=loss_fn,
        lr=lr,
        weight_decay=weight_decay,
        num_epochs=num_epochs,
        device=device,
        dt_ms=dt_ms,
    )

    # Extract the best validation F1 score
    best_val_f1 = metrics['best_val_f1']

    # Return the best validation F1 score for optimization
    return best_val_f1

study = optuna.create_study(direction="maximize")

study.optimize(
    lambda trial: objective(
        trial,
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        loss_fn=SF.ce_rate_loss(),
        device="cuda",
        dt_ms=1.0,
        num_classes=6
    ),
    n_trials=10
)


## PaperSNN

In [20]:
def train_model(model, scheduler, optimizer, train_dataloader, val_dataloader, num_classes=6, lr=1e-3, weight_decay=None, 
                num_epochs=50, model_savepath=None, device="cuda", dt_ms=1.0, class_weights=None, factor=10):
    # Initialize model
    # model = SNNClassifier(num_classes=num_classes, time_steps=4).to(device)
    loss_fn = CustomLoss(num_classes=num_classes, class_weights=class_weights)
    
    # Optimizer
    # optimizer = torch.optim.AdamW(
    #     model.parameters(), 
    #     lr=lr,
    #     weight_decay=weight_decay if weight_decay else 0,
    #     betas=(0.9, 0.999)
    # )
    
    # # Scheduler
    # scheduler = torch.optim.lr_scheduler.LambdaLR(
    #     optimizer,
    #     # lr_lambda=lambda epoch: 0.5 * (1 + np.cos((epoch / num_epochs) * np.pi)) 
    #     if epoch > 0 else 1.0
    # )

    # Updated metrics tracking dictionary (similar to train_model2)
    metrics = {
        'train': {
            'loss': [],           # Per batch loss
            'avg_loss': [],       # Per epoch average loss
            'acc': [],            # Per epoch accuracy
            'f1': [],             # Per epoch overall F1 (if needed)
            'class_metrics': [],  # Per epoch class-specific metrics (if needed)
        },
        'val': {
            'loss': [],
            'avg_loss': [],
            'acc': [],
            'f1': [],
            'class_metrics': [],
        },
        # Spike statistics
        'spike_stats': {
            'train': {
                'avg_spikes_per_neuron': [],
                'spike_rate_hz': [],
                'spike_count': [],
                'active_neurons_percent': [],
                'threshold_proximity_avg': [],
                'threshold_proximity_std': [],
                'membrane_potential_avg': [],
                'membrane_potential_std': [],
            },
            'val': {
                'avg_spikes_per_neuron': [],
                'spike_rate_hz': [],
                'spike_count': [],
                'active_neurons_percent': [],
                'threshold_proximity_avg': [],
                'threshold_proximity_std': [],
                'membrane_potential_avg': [],
                'membrane_potential_std': [],
            },
            'firing_rate_stability': [], # General stability metric
        },
    }


    best_val_f1 = -1
    best_model_state = None
    neuron_cache = {'conv1': None, 'conv2': None}
    threshold_fn = CubicSplineThreshold(scaling_factor=factor)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # --- TRAINING PHASE ---
        model.train()
        
        # Per-epoch tracking
        epoch_data = {
            'train_loss': 0,
            'train_correct': 0,
            'train_total': 0,
            'spike_count': 0,
            'active_neurons': 0,
            'total_neurons': 0,
            'total_possible': 0,
            'train_preds': [],
            'train_targets': [],
            'membrane_sum': 0.0,
            'membrane_sq_sum': 0.0,
            'total_mem_samples': 0,
            'proximity_sum': 0.0,
            'proximity_sq_sum': 0.0,
            'proximity_samples': 0,
        }

        for data, targets in tqdm(train_dataloader, desc="Training"):
            data, targets = data.to(device), targets.to(device)
            utils.reset(model)
            
            # Forward pass
            outputs = model(data)
            spk1, spk2 = model.spk1, model.spk2

            # Cache neuron counts on first batch
            if neuron_cache['conv1'] is None:
                with torch.no_grad():
                    neuron_cache['conv1'] = spk1[0, 0].numel()  # e.g., 16*8*8
                    neuron_cache['conv2'] = spk2[0, 0].numel()  # e.g., 32*4*4

            # Get cached values (conv layers only)
            nl1 = neuron_cache['conv1']
            nl2 = neuron_cache['conv2']
            
            # Dynamic threshold calculation
            thresh_conv1 = threshold_fn.compute_threshold(model.mem1)
            thresh_conv2 = threshold_fn.compute_threshold(model.mem2)

            # Loss calculation
            loss = loss_fn(outputs, targets)
            epoch_data['train_loss'] += loss.item()
            metrics['train']['loss'].append(loss.item())

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.conv1.parameters(), thresh_conv1)
            torch.nn.utils.clip_grad_norm_(model.conv2.parameters(), thresh_conv2)
            optimizer.step()

            # --- METRICS CALCULATION ---
            with torch.no_grad():
                batch_size = data.size(0)
                time_steps = model.time_steps
                
                # Accuracy
                predicted = torch.argmax(outputs, dim=1)
                epoch_data['train_correct'] += (predicted == targets).sum().item()
                epoch_data['train_total'] += targets.size(0)
                
                # Store predictions for F1 calculation
                epoch_data['train_preds'].append(predicted.cpu())
                epoch_data['train_targets'].append(targets.cpu())
                
                # Total neurons calculation (conv layers only)
                total_neurons_batch = (nl1 + nl2) * batch_size * time_steps
                epoch_data['total_neurons'] += total_neurons_batch
                
                # Spike counts (conv layers only)
                batch_spike_count = (spk1.sum() + spk2.sum()).item()
                epoch_data['spike_count'] += batch_spike_count
                
                # Active neurons calculation (conv layers only)
                active_conv1 = (spk1.sum(dim=0) > 0).sum().item()
                active_conv2 = (spk2.sum(dim=0) > 0).sum().item()
                epoch_data['active_neurons'] += active_conv1 + active_conv2

                # Total active neurons possible
                total_active_neurons_possible = (nl1 + nl2) * batch_size
                epoch_data['total_possible'] += total_active_neurons_possible

                # Membrane stats (conv layers only)
                mem = torch.cat([model.mem1.flatten(), model.mem2.flatten()])
                epoch_data['membrane_sum'] += mem.sum().item()
                epoch_data['membrane_sq_sum'] += (mem**2).sum().item()
                epoch_data['total_mem_samples'] += mem.numel()
                
                # Dynamic threshold proximity calculation
                prox_conv1 = torch.abs(model.mem1 - model.conv1.lif.threshold)
                prox_conv2 = torch.abs(model.mem2 - model.conv2.lif.threshold)
                proximity = torch.cat([prox_conv1.flatten(), prox_conv2.flatten()])
                epoch_data['proximity_sum'] += proximity.sum().item()
                epoch_data['proximity_sq_sum'] += (proximity**2).sum().item()
                epoch_data['proximity_samples'] += proximity.numel()

        # --- EPOCH STATISTICS ---
        avg_train_loss = epoch_data['train_loss'] / len(train_dataloader)
        metrics['train']['avg_loss'].append(avg_train_loss)
        
        train_acc = epoch_data['train_correct'] / epoch_data['train_total'] if epoch_data['train_total'] > 0 else 0
        metrics['train']['acc'].append(train_acc)
        
        # Calculate F1 and class metrics if needed
        train_preds = torch.cat(epoch_data['train_preds']).numpy() if epoch_data['train_preds'] else np.array([])
        train_targets = torch.cat(epoch_data['train_targets']).numpy() if epoch_data['train_targets'] else np.array([])
        
        
        train_f1 = f1_score(train_targets, train_preds, average="weighted") if len(train_preds) > 0 else 0.0
        
        # Calculate class-specific metrics
        if len(train_preds) > 0:
            train_class_metrics = calculate_class_metrics_snn(train_targets, train_preds)
            metrics['train']['class_metrics'].append(train_class_metrics)
        else:
            train_class_metrics = {
                'benign': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0},
                'malicious': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
            }
            
        metrics['train']['f1'].append(train_f1)
        metrics['train']['class_metrics'].append(train_class_metrics)
        
        # Calculate spike statistics
        avg_spikes = epoch_data['spike_count'] / epoch_data['total_neurons'] if epoch_data['total_neurons'] > 0 else 0
        active_percent = (epoch_data['active_neurons'] / epoch_data['total_possible']) * 100 if epoch_data['total_possible'] > 0 else 0

        # Calculate membrane potential statistics
        mem_avg = epoch_data['membrane_sum'] / epoch_data['total_mem_samples'] if epoch_data['total_mem_samples'] > 0 else 0
        mem_std = np.sqrt(epoch_data['membrane_sq_sum'] / epoch_data['total_mem_samples'] - mem_avg**2) if epoch_data['total_mem_samples'] > 0 else 0
        
        # Calculate threshold proximity statistics
        prox_avg = epoch_data['proximity_sum'] / epoch_data['proximity_samples'] if epoch_data['proximity_samples'] > 0 else 0
        prox_std = np.sqrt(epoch_data['proximity_sq_sum'] / epoch_data['proximity_samples'] - prox_avg**2) if epoch_data['proximity_samples'] > 0 else 0

        # Update spike statistics history
        metrics['spike_stats']['train']['avg_spikes_per_neuron'].append(avg_spikes)
        metrics['spike_stats']['train']['spike_rate_hz'].append(avg_spikes * (1000/dt_ms))
        metrics['spike_stats']['train']['spike_count'].append(epoch_data['spike_count'])
        metrics['spike_stats']['train']['active_neurons_percent'].append(active_percent)
        metrics['spike_stats']['train']['membrane_potential_avg'].append(mem_avg)
        metrics['spike_stats']['train']['membrane_potential_std'].append(mem_std)
        metrics['spike_stats']['train']['threshold_proximity_avg'].append(prox_avg)
        metrics['spike_stats']['train']['threshold_proximity_std'].append(prox_std)

        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2%}")
        print(f"Spike Rate: {avg_spikes*(1000/dt_ms):.1f}Hz | Active Neurons: {active_percent:.1f}%")
        print(f"Membrane Potential: {mem_avg:.4f} ± {mem_std:.4f}")
        print(f"Threshold Proximity: {prox_avg:.4f} ± {prox_std:.4f}")

        # --- VALIDATION PHASE ---
        with torch.no_grad():
            model.eval()
            
            # Per-epoch validation tracking
            val_data = {
                'val_loss': 0,
                'val_correct': 0,
                'val_total': 0,
                'spike_count': 0,
                'active_neurons': 0,
                'total_neurons': 0,
                'total_possible': 0,
                'val_preds': [],
                'val_targets': [],
                'membrane_sum': 0.0,
                'membrane_sq_sum': 0.0,
                'total_mem_samples': 0,
                'proximity_sum': 0.0,
                'proximity_sq_sum': 0.0,
                'proximity_samples': 0,
            }
            
            for data, targets in tqdm(val_dataloader, desc="Validation"):
                data, targets = data.to(device), targets.to(device)
                utils.reset(model)
                
                outputs = model(data)
                spk1, spk2 = model.spk1, model.spk2
                
                # Loss and accuracy
                loss = loss_fn(outputs, targets)
                val_data['val_loss'] += loss.item()
                metrics['val']['loss'].append(loss.item())

                # Accuracy
                predicted = torch.argmax(outputs, dim=1)
                val_data['val_correct'] += (predicted == targets).sum().item()
                val_data['val_total'] += targets.size(0)
                
                # Store predictions for F1 calculation
                val_data['val_preds'].append(predicted.cpu())
                val_data['val_targets'].append(targets.cpu())

                # Spike statistics
                batch_size = data.size(0)
                time_steps = model.time_steps
                nl1 = neuron_cache['conv1']
                nl2 = neuron_cache['conv2']

                # Total neurons (conv layers only)
                total_neurons_batch = (nl1 + nl2) * batch_size * time_steps
                val_data['total_neurons'] += total_neurons_batch
                
                # Spike counts (conv layers only)
                val_data['spike_count'] += (spk1.sum() + spk2.sum()).item()
                
                # Active neurons (conv layers only)
                active_conv1 = (spk1.sum(dim=0) > 0).sum().item()
                active_conv2 = (spk2.sum(dim=0) > 0).sum().item()
                val_data['active_neurons'] += active_conv1 + active_conv2
                
                # Total active neurons possible
                total_active_neurons_possible = (nl1 + nl2) * batch_size
                val_data['total_possible'] += total_active_neurons_possible

                # Membrane stats
                mem = torch.cat([model.mem1.flatten(), model.mem2.flatten()])
                val_data['membrane_sum'] += mem.sum().item()
                val_data['membrane_sq_sum'] += (mem**2).sum().item()
                val_data['total_mem_samples'] += mem.numel()
                
                # Dynamic threshold proximity calculation
                prox_conv1 = torch.abs(model.mem1 - model.conv1.lif.threshold)
                prox_conv2 = torch.abs(model.mem2 - model.conv2.lif.threshold)
                proximity = torch.cat([prox_conv1.flatten(), prox_conv2.flatten()])
                val_data['proximity_sum'] += proximity.sum().item()
                val_data['proximity_sq_sum'] += (proximity**2).sum().item()
                val_data['proximity_samples'] += proximity.numel()

        # --- VALIDATION METRICS ---
        avg_val_loss = val_data['val_loss'] / len(val_dataloader)
        val_acc = val_data['val_correct'] / val_data['val_total'] if val_data['val_total'] > 0 else 0
        
        metrics['val']['avg_loss'].append(avg_val_loss)
        metrics['val']['acc'].append(val_acc)

        # Calculate F1 and class metrics if needed
        val_preds = torch.cat(val_data['val_preds']).numpy() if val_data['val_preds'] else np.array([])
        val_targets = torch.cat(val_data['val_targets']).numpy() if val_data['val_targets'] else np.array([])
        
        
        # Overall F1 score (binary average)
        val_f1 = f1_score(val_targets, val_preds, average='weighted') if len(val_preds) > 0 else 0.0
        metrics['val']['f1'].append(val_f1)
        
        # Calculate class-specific metrics
        if len(val_preds) > 0:
            val_class_metrics = calculate_class_metrics_snn(val_targets, val_preds)
            metrics['val']['class_metrics'].append(val_class_metrics)
        else:
            val_class_metrics = {
                'benign': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0},
                'malicious': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
            }
            metrics['val']['class_metrics'].append(val_class_metrics)


        # Calculate validation spike statistics
        val_avg_spikes = val_data['spike_count'] / val_data['total_neurons'] if val_data['total_neurons'] > 0 else 0
        val_active_percent = (val_data['active_neurons'] / val_data['total_possible']) * 100 if val_data['total_possible'] > 0 else 0
        
        # Calculate membrane statistics
        val_mem_avg = val_data['membrane_sum'] / val_data['total_mem_samples'] if val_data['total_mem_samples'] > 0 else 0
        val_mem_std = np.sqrt(val_data['membrane_sq_sum'] / val_data['total_mem_samples'] - val_mem_avg**2) if val_data['total_mem_samples'] > 0 else 0
        
        # Calculate threshold proximity statistics
        val_prox_avg = val_data['proximity_sum'] / val_data['proximity_samples'] if val_data['proximity_samples'] > 0 else 0
        val_prox_std = np.sqrt(val_data['proximity_sq_sum'] / val_data['proximity_samples'] - val_prox_avg**2) if val_data['proximity_samples'] > 0 else 0

        # Update validation spike statistics history
        metrics['spike_stats']['val']['avg_spikes_per_neuron'].append(val_avg_spikes)
        metrics['spike_stats']['val']['spike_rate_hz'].append(val_avg_spikes * (1000/dt_ms))
        metrics['spike_stats']['val']['spike_count'].append(val_data['spike_count'])
        metrics['spike_stats']['val']['active_neurons_percent'].append(val_active_percent)
        metrics['spike_stats']['val']['membrane_potential_avg'].append(val_mem_avg)
        metrics['spike_stats']['val']['membrane_potential_std'].append(val_mem_std)
        metrics['spike_stats']['val']['threshold_proximity_avg'].append(val_prox_avg)
        metrics['spike_stats']['val']['threshold_proximity_std'].append(val_prox_std)

        # Model checkpointing
        # use f1 score as the metric to save the best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'val_loss': avg_val_loss,
                'val_acc': val_acc,
                'val_f1': val_f1,
            }
            print(f"New best model found! Val F1: {val_f1:.4f}")

        print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2%} | Val F1: {val_f1:.4f}")
        print(f"Val Spike Rate: {val_avg_spikes*(1000/dt_ms):.1f}Hz | Active Neurons: {val_active_percent:.1f}%")
        print(f"Benign - P: {val_class_metrics['benign']['precision']:.4f}, R: {val_class_metrics['benign']['recall']:.4f}, F1: {val_class_metrics['benign']['f1']:.4f}")
        print(f"Malicious - P: {val_class_metrics['malicious']['precision']:.4f}, R: {val_class_metrics['malicious']['recall']:.4f}, F1: {val_class_metrics['malicious']['f1']:.4f}")
        print(f"Membrane Potential: Train {mem_avg:.4f} ± {mem_std:.4f} | Val {val_mem_avg:.4f} ± {val_mem_std:.4f}")
        print(f"Threshold Proximity: Train {prox_avg:.4f} ± {prox_std:.4f} | Val {val_prox_avg:.4f} ± {val_prox_std:.4f}")
        print("-" * 50)

        # Update scheduler
        scheduler.step()

    # Final saving
    if model_savepath:
        final_state = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': metrics
        }
        torch.save(final_state, model_savepath)
        
        if best_model_state is not None:
            best_path = model_savepath.replace(".pt", "_best.pt")
            torch.save(best_model_state, best_path)
            print(f"Best model (F1: {best_val_f1:.4f}) saved at {best_path}")

    return model, metrics

In [24]:
from scipy.interpolate import CubicSpline

In [25]:
class CubicSplineThreshold:
    def __init__(self, scaling_factor=10):  # Document-specified default
        self.scaling_factor = scaling_factor

    def compute_threshold(self, mem_rec):
        """
        Pure implementation of document's Section 3.2 and Equations 7-10
        mem_rec: (T, B, C, H, W) membrane potentials
        """
        # Document specifies using first sample only
        mem_sample = mem_rec[:, 0].detach().flatten().cpu().numpy()
        
        # Document's exact spline fitting logic (Eq. 7-8)
        x = np.arange(len(mem_sample))
        cs = CubicSpline(x, mem_sample)
        
        # Document's slope calculation (Eq. 9-10)
        derivatives = cs(x, 1)  # First derivative
        avg_slope = np.mean(np.abs(derivatives))
        
        # Document-specified clipping (Sec 3.2)
        return float(np.clip(avg_slope * self.scaling_factor, 0.1, 5.0))

In [ ]:
def objective(trial, train_dataloader, val_dataloader, device="cuda", dt_ms=1.0, num_classes=2):
    # Define hyperparameters to optimize
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)  # Learning rate
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)  # Weight decay
    num_epochs = trial.suggest_int("num_epochs", 10, 100)  # Number of epochs
    beta1 = trial.suggest_float("beta1", 0.8, 0.999)
    beta2 = trial.suggest_float("beta2", 0.9, 0.9999)
    beta_model = trial.suggest_float("beta_model", 0.1, 0.9)
    num_steps = trial.suggest_int("num_steps", 2, 8)  # Number of time steps
    alpha = trial.suggest_float("alpha", 0.1, 0.9)  # Alpha for the LIF neuron
    scaling_factor = trial.suggest_int("scaling_factor", 5, 20)  # Scaling factor for the threshold

    model = SNNClassifier(time_steps=num_steps, num_classes=num_classes, threshold=beta_model, alpha=alpha).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(beta1, beta2))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: 0.5 * (1 + np.cos((epoch / num_epochs) * np.pi)) if epoch > 0 else 1.0)


    # Train the model using the provided training function
    model, metrics = train_model(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        lr=lr,
        weight_decay=weight_decay,
        num_epochs=num_epochs,
        device=device,
        dt_ms=dt_ms,
        factor=scaling_factor,
        num_classes=num_classes
    )

    # Extract the best validation F1 score
    best_val_f1 = metrics['best_val_f1']

    # Return the best validation F1 score for optimization
    return best_val_f1

study = optuna.create_study(direction="maximize")

study.optimize(
    lambda trial: objective(
        trial,
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        device="cuda",
        dt_ms=1.0,
        num_classes=6
    ),
    n_trials=10
)
